# 画像のノイズ削除

\label{sec:denoise}

## 問題設定

実験で画像を出力する測定を行うとノイズが入ることが多くあります．ノイズの入り方は様々です．
元画像とノイズの特徴が全く異なる場合，例えば，推薦システムで用いた手法でノイズが除去できることが理解できると思います．
また，ニューラルネットワークモデルでも画像からノイズを除去すること（デノイズ）が可能です．
一般的には，ノイズが含まれない画像とそれにノイズを追加した画像が大量に組で与えられデノイズを行うモデル学習を行うことが多いようです．この場合は，画像の品質を改善し，重要なディテールを失わないようにすることが可能です．
しかし，物質科学の測定ではそのような大量が画像セットが与えられていることは稀で，ノイズが含まれた画像データセットしか存在しないことが多いと思います．
この場合に画像の品質は落ちてしまうかもしれませんが，「ノイズが含まれた画像データセットしかない」場合にニューラルネットワークモデルによりノイズを除去することが可能です．
デノイズ手法について
GPT4oから知識を取得しながら，コード生成を行う過程例を紹介します．

## 知識の取得

### ニューラルネットワークモデル学習過程

ニューラルネットワークモデルの学習の知識がほぼ無いとして話を進めます．
世の中には名前が付いているものから付いていないものまで様々なニューラルネットワークモデルが提案されています．
まず，ニューラルネットワークについて一般的な知識として，ニューラルネットワークの学習過程についてGPT4oに説明を依頼します．

GPT4oに以下を尋ねます．
```
プロンプト「
ニューラルネットワークモデルの学習過程について教えて下さい．
」
```

下は回答を要約して書いています．

---（回答）---

### 学習過程

1. データの準備として説明変数と目的変数の組，そして訓練セットでモデル学習し，テストセットでモデルの性能を評価する．
2. 複数の「層」で構成され，最初に，これらの重みとバイアスをランダムに初期化しますニューラルネットワークモデルを用意する．
3. フォワードプロパゲーション（順伝播）：
トレーニングデータをネットワークに入力し，各層で計算を行いながら次の層へデータを伝達し，最終的に，モデルの出力層で予測結果が得られる．
層での計算は，基本的には線形変換（重みとバイアスを使って計算）と非線形活性化関数（例: ReLUやシグモイド関数）によって行われる．
4. **損失関数**の計算：
モデルの予測値と目的変数の正解値から損失関数により損失を計算する．回帰問題では「平均二乗誤差（MSE）」，分類問題では「クロスエントロピー損失」などが使われる．
5. バックプロパゲーション（逆伝播）：
損失関数に対する重みの微分である勾配を計算し，勾配降下法（Gradient Descent）などの最適化アルゴリズムがを使用し，モデルのパラメータ（重みとバイアス）を更新する．
6. 重みの更新：
逆伝播に基づいて，ニューラルネットワークの重みを更新する．
$$
w = w - \eta \frac{\partial L}{\partial w}
$$
ここで，$w$ は重み，$\eta$ は学習率，$L$ は損失関数，$\frac{\partial L}{\partial w}$ は重みに対する損失関数の勾配です．

7. 反復：
「**エポック（epoch）**」と呼ぶ，フォワードプロパゲーションから重みの更新までのプロセスを，データセット全体に対して複数回繰り返す．
8. 最後にモデルの性能を評価し，モデルのパラメタを調整し，性能を最適化する．

   
\noindent
---（回答ここまで）---

残念ながら，「バックプロパゲーション（逆伝播）」と「重みの更新」は内容が重複しています．しかし，ここではGPT4oの回答の過程は変えずに載せています．
この時点で分からないことがあれば，それらをGPT4oに尋ねてください．
例えば，同じChatで「エポックとは何ですか？」とGPT4oに尋ねた場合の回答を要約すると，
「エポックとは，トレーニングデータセット全体を使って一度の学習プロセスを完了すること，一つのエポックが完了するたびに，モデルは全てのトレーニングデータを一通り学習したことになる」という回答を得られます．


### デノイズ手法

デノイズの問題に戻ります．以下の流れの見通しを良くするために，手法を選択するまでの過程を図\ref{fig:denoise_method_selection}に示します．


![デノイズ手法選択過程](image_keep/LLM-development-process-step1.png){ width=10cm }

\label{fig:denoise_method_selection}


まず，ニューラルネットワークモデルを利用したデノイズ手法に全く知識が無いとしてそれらの手法についてGPT4oに尋ねます．図
図\ref{fig:denoise_method_selection}で，背景が灰色の過程を最初に行います，

```
プロンプト「
# 私とあなたの立場
私は機械学習手法の初心者です．
あなたは機械学習手法のエキスパートです．あなたは優しく機械学習手法を教えることができます．
# 依頼
 画像からノイズを除去するニューラルネットワークモデルを利用した手法について教えてください．
# 条件
ノイズが含まれた画像データセットしかありません．
」
```

GPT4oはクリーンな画像が不要なデノイジング手法として，以下の三つを紹介します．特徴を要約して以下に記します．

---（回答まとめ）---

1. Noise2Noise：
同じシーンの異なるノイズが含まれた複数の画像を使用してモデルを訓練します．
ノイズがランダムであれば，モデルはクリーンな信号（ノイズ以外の部分）を学習し，ノイズを除去する能力を獲得します．

2. Noise2Void / Noise2Self：
単一のノイズ画像からノイズを除去する自己教師あり学習（self-supervised learning）の手法です．
できれば，同じシーンの異なるノイズ画像が複数あると良いですが，単一の画像でも自己教師あり手法は可能です．

3. Blind-Spot Networks：
Blind-Spot Networksは，特定の画素の値を予測する際に，その画素自体の情報を使用しないネットワークです．これにより，ノイズが直接影響しないようにします．

\noindent
そして，**Noise2Self**や**Noise2Void**を初心者の方に勧め，簡単なコードを同時に出力しました．

---（回答まとめここまで）---


Noise2Selfのアルゴリズムの特徴を更にGPT4oに尋ねると以下の三つの特徴を答えます．

---（回答まとめ）---

1. マスク付き入力画像の生成：画像の一部をランダムに隠し，学習に入れない．
2. 損失関数：マスクされていない部分から損失関数を計算する．
3. ネットワークの学習：マスクされていない部分のみからマスクされた部分の画素値を予測する．

---（回答まとめここまで）---

ただし，調べてみると，GPT4oが説明するNoise2Selfのアルゴリズムは，その考え方のごく一部を用いたものでNoise2Self\cite{Noise2Self}やNoise2Void\cite{Noise2Void}そのものではありませんでした．（図\ref{fig:denoise_method_selection}①の過程）


他の手法を探すために異なる質問をします．（図\ref{fig:denoise_method_selection}②の過程）

```
プロンプト「
デノイズに使えるニューラルネットワークモデルを教えて下さい．

# 手法の概要

    手法を簡潔に説明してください．
    手法の目的を説明してください．
    手法の背景にある仮説を説明してください．

# 手法の詳細

    手法の概要・構成要素を説明してください．
    手法の概要・構成要素に紐づけられる基本的な数式や理論を説明してください．
    手法の長所，短所を説明してください．
    手法が機械学習モデルである場合にハイパーパラメタを教えてください．
    コードの説明ではなく手法について教えてください．
」
```

---（回答まとめ）---

GPT4oはU-Netを薦めます．しかし，U-Netを調べると一般的には4から5隠れ層数でフィルター数がエンコーダー部分で64, 128, 256, 512と増えていく畳み込みニューラルネットワークであると回答します．（図\ref{fig:denoise_method_selection}③の過程）

---（回答まとめここまで）---

GPU無しで実行する本書の例としては実行時間がかかりすぎるので，より少ない隠れ層のネットワークで問題を解決できないでしょうか．

```
プロンプト「
二層程度の隠れ層を持つ簡単なニューラルネットワークでデノイズに使えるモデルはありますか？
」
```

GPT4oは二つの隠れ層を持つオートエンコーダーを薦めました．（図\ref{fig:denoise_method_selection}④の過程）
説明をまとめると，

---（回答まとめ）---

- これはエンコーダー部分とデコーダー部分に畳み込み層を用いるオートエンコーダーであること．
- 損失関数として例えばMSEを用いること．
- 長所としては実装が容易なこと，計算コストが低いこと，ノイズ除去に有効なこと
- 短所としては複雑なデータ構造に対してはパフォーマンスが限られること，過学習しやすい傾向があること，高度な特徴抽出には限界があること
  
等が示されます．また，典型的なハイパーパラメタについても表示されます．

---（回答まとめここまで）---


## 指示書の作成

GPT4oにコードの作成を依頼することはコードの外注をするようなものです．
外注先にコードの作成を依頼する場合に，大雑把な指示だと依頼された業者も何を行えば良いのか困りますし，指示通りのコードだとしても発注側は納品されたコードを利用して目的を達成するには大きな修正が必要になるでしょう．
ある目的を達成できるコード実装は一般的に様々ですが，コード生成は膨大な可能性から一つの実装を選択することです．
私はデノイズコードを生成するために幾つかの方法を試しました．
その試行の範囲内では，ユーザーがあらかじめ手法をある程度選択して，コード生成の可能性を絞っておいた方が上手くいきましたのでその手法をここで用います．
（ある程度）詳細な指示書をGPT4oに作成させてからコードを作成し，実行する過程を行います．
この過程をまとめて図\ref{fig:denoise_instruction}に記します．
以下ではGPT4oにコード生成を依頼する「指示書」とその全段階の「指示書案」を分けて書いています．

- ①ユーザーが簡単に目的と条件を書いてGPT4oに指示書案の作成を依頼します．
- ②GPT4oが詳細な指示書案を書きます．
- ③GPT4oと会話しながら手法について学習し，機能選択・追加を行います．
- ④指示書を完成させGPT4oにコードを生成させます．簡単なエラーはGPT4oにエラーメッセージを書くことでGPT4oは修正ができます．
- ⑤実行時に初めて足りない機能があることもわかるでしょう．それらを追加します．また動作不良が起きた部分はユーザーが，もしくはGPT4oと会話しながらコードを完成させます．


![指示書作成と実行](image_keep/LLM-development-process-step2.png){ width=10cm }

\label{fig:denoise_instruction}

入力画像として
TEM画像(ディレクトリcropped_64x64)を用意しています．この画像自体は別途生成AIで生成しています．(GPT4oの回答に合わせて配列(64,64)を64x64と書きます．）
このTEM画像に予め輝度が0もしくは最大(255)の値のランダムノイズを割合パーミル値$X$加えたTEM画像(ディレクトリcropped_64x64_permilX)を用意しています．

---（回答まとめ）---

なお，このようなノイズを何と言うかもGPT4oに尋ねるとソルトアンドペッパーノイズと回答が得られます．ソルト（明るい）ペッパー（暗い）部分ということもGPT4oが説明してくれます．また．ソルトアンドペッパーノイズに対してはニューラルネットワークを用いなくても，簡単かつ強力なデノイズアルゴリズムが知られています．
これもまたGPT4oが回答でき，例えば，周囲の画素値の中央値を取ることでノイズを軽減するメディアンフィルタがscipy.imageにすでに用意されています．

---（回答まとめここまで）---

しかし，ここではニューラルネットワークを用いてノイズの軽減を行います．

まずは大まかな指示を書きGPT4oに「指示書案」を書いてもらいます．（図\ref{fig:denoise_instruction}①の過程）
ここで，Pytorchを用いるように明示します．
有名なニューラルネットワークフレームワークにKerasとPytorchがありますが，Pytorchの方が小規模PCでは高速に動作するためです．

```
プロンプト「
# 依頼
グレイスケール画像のデノイズプログラムを外注するため指示書案を書いてください．
指示書案ではプログラム詳細部分のみ書いてください．
プログラムを書かないでください．
自然言語で説明をしてください．
依頼の詳細は{# 依頼詳細}に従ってください．
指示書を作成する上での不明点は私に尋ねてください．

# 依頼詳細
二つの隠れ層を持つオートエンコーダーを用いてください．
PythonとPytorchを用いてください．
画像一枚ごとにモデル学習しデノイズします．画像サイズは64x64です．画像ファイル名はimg_filenameで指定します．
一枚ずつデノイズを行ってください．
」
```

GPT4oはこれに対して詳細な過程を加えることができます．（図\ref{fig:denoise_instruction}②の過程）
回答の中で詳細な過程部分を以下に記載します．

--（回答)---


#### インポートと初期設定

- torch, torchvisionなどの必要なPythonライブラリをインポートすること.
- `device` 変数を定義し，可能な場合は GPU を使用してください．例えば，`torch.device("cuda" if torch.cuda.is_available() else "cpu")`．

#### データロード

- 画像読み込みは `img_filename` 変数で指定されたファイルパスから行います．
- 画像を読み込み，正規化し，適切なテンソル形式（64x64のサイズ）に変換してください．これには `torchvision.transforms` を使用します．

#### オートエンコーダーモデルの定義

- 二層の隠れ層を持つデノイズオートエンコーダーを定義してください．
1. **エンコーダー:**
 - 第一の隠れ層には入力から隠れ層への線形変換（完全結合層）とReLU活性化関数が含まれます．
 - 第二の隠れ層も同様に線形変換とReLU活性化関数を含みます．
2. **デコーダー:**
 - 第二の隠れ層から中間の隠れ層への線形変換とReLU活性化関数．
 - 中間の隠れ層から出力への線形変換（完全結合層）

- 具体的な実装は，PyTorchの `nn.Module` クラスを継承して行います．

#### トレーニングの設定

- 損失関数には，平均二乗誤差損失（MSELoss）を使用してください．
- オプティマイザには，Adamオプティマイザを使用し，学習率を指定してください（例: `lr=0.001`）．
- 一般的なトレーニングのエポック数やバッチサイズについても設定してください（デフォルト値：エポック数`20`，バッチサイズ`32`）．

#### トレーニングループ

- ノイズ付きの画像とクリーンな画像から成る訓練データセットを準備してください．
- 毎エポックごとに以下の手順を実行してください．
  1. モデルの予測を実行します．
  2. 損失を計算します．
  3. 勾配をゼロにリセットします（`optimizer.zero_grad()`）．
  4. 逆伝播を実行し，勾配を計算します（`loss.backward()`）．
  5. オプティマイザをステップ実行し，パラメータを更新します（`optimizer.step()`）．

#### デノイズ処理

- トレーニングが完了した後，指定された `img_filename` の画像をモデルに通し，デノイズされた画像を生成してください．
- デノイズされた画像は，適切なフォーマットに変換し，保存してください．推奨されるフォーマットはPNGです．

#### 評価

- デノイズされた結果を元の画像と比較して評価するためのコード（任意）を追加してください．評価指標にはPSNR（ピーク信号対雑音比）やSSIM（構造類似性インデックス）を使用します．

\noindent
---（回答終わり)---


GPT4oはかなり詳細に過程を補完してくれました．トレーニングループは既にGPT4oに尋ねた部分を簡単に説明しています．
分からない部分はGPT4oに聞いて理解することを検討してください．
しかし，もし，この段階で分からなくても，コードを見ながら再びGPT4oに尋ねると理解できるかもしれません．

指示書を完成させましょう．私は不要と思う部分は削除しました．
例えば，画像をファイルとして保存する機能は不要としました．また，評価指標の出力は（この時点では）必要性が理解できないので削除しました．
追加機能として，画像は一定のエポック数毎に出力させ，最後に学習過程を示すように書き直しました．
（図\ref{fig:denoise_instruction}③の過程）
この際にGPT4o以外で項目の妥当性の確認をすることも検討してください．

## コード生成と修正

### コード生成

\label{sec:code-generation}

では，この指示書でGPT4oにPythonコード作成依頼をします．（図\ref{fig:denoise_instruction}④の過程）

```
プロンプト「
# 依頼
グレイスケール画像のデノイズプログラムを作成してください．
依頼の詳細は{# 依頼詳細}に従ってください．
指示書を作成する上での不明点は私に尋ねてください．

# 依頼詳細
PythonとPytorchを用いてください．
画像一枚ごとにモデル学習しデノイズします．画像サイズは64x64です．
学習率，エポック数などのモデルのハイパーパラメタは変数として定義しユーザーが設定可能にしてください．

## データローダーの作成
画像ファイルは変数img_filenameで与えられます．
画像データを0から1までに正規化してください．

## モデルの定義
二層のデノイズ用のオートエンコーダーを用いてください．

## 損失関数とオプティマイザーを設定してください．
損失関数としてMSE（Mean Squared Error）を使用します．
オプティマイザとしてAdamかSGDを使用します．

##トレーニングループ
一定のエポック毎に損失値を表示させ，トレーニングの進捗を確認してください．
一定のエポック毎にデノイズされた画像と元画像を比較してください．

## 学習後
Jupyter Notebookを用いるのでデノイズされた画像をファイルとして保存する必要はありません．
最後にエポック vs 損失関数を図示してください．
」
```

ここで
Pythonコードを作成し
GPT4oはニューラルネットワークを用いた学習を行うコードの一般的な形を提示してくれます．私の実行した際は以下の形になっていました．


---（回答）---

```
class ImageDataset(Dataset):
    (画像を読み込むクラスの定義）
     
dataset = ImageDataset(img_filename, ...)
dataloader = DataLoader(dataset, batch_size=batch_size, ...)

class Autoencoder(nn.Module):
    (ニューラルネットワークモデルの定義）

model = Autoencoder()

# 損失関数とオプティマイザーの定義
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    for data in dataloader:
        img = data
        output = model(img) 
        loss = criterion(output, img)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```
---（回答終わり）---


GPT4oはコードの説明が得意です．
ニューラルネットワークモデルの学習過程と対比させて説明してもらいます．
{# 過程}の「詳細な過程部分」というのは\ref{sec:code-generation}のGPT4oの回答を指します．


```
プロンプト「
{# コード}に対して{# 過程}がどこにあるのか対比させて説明をしてください．

# 過程
（詳細な過程部分を貼り付け）

# コード
（コードを貼り付け）
」
```

以下は回答のまとめです．一回の回答ではなく，一部は詳細を再度尋ねてまとめています．

---（回答まとめ）---

#### データローダーの定義

'dataloader = DataLoader(dataset, batch_size=batch_size, ...)'はDataLoader は，データセットを扱いやすい形でミニバッチに分割して提供します．
batch_size は一度にロードするデータの量を指定します．
（このプログラムでは一枚の画像なので必ず1づつ提供されます．）

#### 損失関数とオプティマイザーの定義

`'criterion = nn.MSELoss()'
は復元された画像と元の画像の間の誤差（二乗平均平方誤差）を計算する損失関数を定義します．
'optimizer = optim.Adam(model.parameters(), lr=learning_rate)'は，Adamオプティマイザーを使用してモデルのパラメータを更新することを定義します．


#### フォーワードプロパゲーション

'output = model(img)'は入力データをモデルに通して予測結果を得ます．

#### 損失関数の計算

'loss = criterion(output, img)'は
MSE損失関数を計算します．


#### バックプロパゲーション

'optimizer.zero_grad()' で勾配を初期化し，'loss.backward()' で勾配を計算します．

#### 重みの更新

'optimizer.step()'はオプティマイザーを使ってモデルの重みを更新します．


#### 反復（エポック）

'for epoch in range(num_epochs):' は，全訓練エポック数を指定します．
'for data in dataloader:' は，データローダーからミニバッチごとにデータを取得します．

---（回答まとめここまで）---


GPT4oの説明はよく整理されており理解も容易と思います．
GPT4oはこの基本的な枠組みに従いコードを生成します．


### エラー修正

以下では，GPT4oによるコード生成（図\ref{fig:denoise_instruction}④の過程）と，実行後の追加機能（図\ref{fig:denoise_instruction}⑤の過程）について説明します．生成されたコードには，主に2つの種類の誤りが存在します．

- 実行時エラー: 実行時エラーはGPT4oに修正依頼します．
例えば，PyTorchのテンソル配列に誤りがあることがよくあります．この場合，エラーメッセージをGPT4oに入力すれば，修正が期待できます．

- コードの論理て誤り：
コードの説明をGPT4oに求め，実行過程を確認してください． 私の経験では，入力画像や出力画像が適切に正規化されていないことがありました．GPT4oは現時点で，生成された内容の整合性を取ることが難しいようです．たとえ，入力画像が0から1までに正規化されていると指示したとしても，デノイズ後の画像が0から1に正規化されていないことがあります．適切に正規化されないと，損失関数の計算が正確に行われません．正規化が0から1の場合，デコーダーの最終層でSigmoid関数（ $\sigma(x) = 1/(1 + e^{-x})$ ）を使用します． -1から1の正規化の場合は，Tanh関数（ $\text{tanh}(x) = (e^x - e^{-x})/(e^x + e^{-x})$ ）を使用します． これらの修正は，GPT4oと相談しながらユーザーが行うか，再度GPT4oに依頼することで実施しました．


私の経験では，正常に動作させるために，GPT4oが生成したコードをユーザーが自ら修正する必要がありました．しかし，GPT4oが作成したニューラルネットワークの基本的な学習コードには，細部のみ修正すればよいので，作業量が大幅に減少します．これは初学者にとっては大きな利点です．上級者は，一般的なコード形式にこだわらず，自分のスタイルで作成したいかもしれません．その場合，自分のコード例を添付すれば，それに従って調整できます． 正常に動作したコードは，8001.1000.denoise_code.ipynbに保存しています．
また，以下の修正を含め，動作を'setting'変数で制御しています．



### ニューラルネットワークモデルの説明

コード生成過程で確認した方が良いことですが，この節で説明します．
ここで用いている
畳み込みオートエンコーダーモデルのAutoencoderクラス
をGPT4oに説明してもらいます．ここでは'2layers_kernelsize3'として選択するAutoencoderクラスを説明対象とします．
このクラスはカーネル，ストライド，パディングを使用しているため，多くの要素を説明できるからです．

```
プロンプト「
以下のAutoencoderクラスをまとめて表にしてください．
入力画像はgray scaleで配列（64，64）の画像を入力とします．

# コード
（kernel,stride, paddingを用いた'2layers_kernelsize3'の場合のAutoencoderクラスを添付する．）
」
```

結果を表\ref{tabl:denoise_autoencoder_class}に示します．

---（回答）---

![1x64x64の入力に対するAutoencoderクラスのネットワーク構造例](table_keep/denose_autoencoder_class_list.png){ width=10cm }

\label{tabl:denoise_autoencoder_class}

---（回答ここまで）---

入力（input)の1x64x64の第一要素の1はグレイスケールのためです．
RGBカラー画像の場合は3になります．

カーネルサイズ，ストライド，パディングといった用語は初学者には難しいかもしれません，GPT4oにこれらの意味を尋ねます．

```
プロンプト「
カーネルサイズ，ストライド，パッディングとは何ですか？
」
```

以下に回答をまとめます，

---（回答）---


カーネルサイズは，畳み込み層におけるフィルター（カーネル）の高さと幅です．
ストライドは，カーネルが入力データに適用された後，次の適用位置に移動する際のステップサイズです，ストライドが1ならば，カーネルは1画素ずつ移動します．
パッディングは，カーネルが入力データの端にも適用できるように，入力データの周囲に画素を追加する方法です，通常はゼロで埋めます．

---（回答ここまで）---

さらに具体例を使って説明することもできます，その場合，簡単な行列でこれらを説明します．
表\ref{tabl:denoise_autoencoder_class}では，第一層が1x64x64から32x64x64に変化しています．
これを行っている'nn.Conv2d'関数について説明してもらいましょう．

```
プロンプト「
# 私
私はニューラルネットワークとPyTorchの初心者です．
# 依頼
nn.Conv2d(1, 32, kernel_size=3, padding=1),
の入出力サイズの計算式を教えてください．
」
```

GPT4oの回答以下にまとめます．

---（回答）---


#### Conv2d の構成

```
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
```

上記の `nn.Conv2d` は以下のパラメータを持ちます：
- `in_channels`: 入力チャネル数 (例: グレースケール画像なら1，カラー画像なら3)
- `out_channels`: 出力チャネル数 (フィルター数)
- `kernel_size`: カーネル（フィルター）のサイズ（例: 3なら3x3のカーネル）
- `stride`: ストライド（デフォルトは1）
- `padding`: パディング（周囲を埋める画素の数，デフォルトは0）

#### 入出力サイズの計算

- 入力の高さ・幅：入力画像の高さと幅
に対して入力サイズから出力サイズを計算するための一般的な公式は以下になります．

$$
\text{出力の高さ} = \left\lfloor \frac{\text{入力の高さ} + 2 \times \text{パディング} - \text{カーネルの高さ}}{\text{ストライド}} \right\rfloor + 1
$$

$$
\text{出力の幅} = \left\lfloor \frac{\text{入力の幅} + 2 \times \text{パディング} - \text{カーネルの幅}}{\text{ストライド}} \right\rfloor + 1
$$

---（回答ここまで）---

ニューラルネットワークモデルの構造を変更するには，これらのパラメータを適切に設定する必要があります．
GPT-4oでは内部でこの計算式を用いることで，容易にニューラルネットワークモデルを変更できます．
8000.1000.denose_code.ipynbには，複数のAutoencoderクラスがあります．
プロンプトに「隠れ層を3層にしてください」「エンコーダーの最初を'nn.Conv2d(1, 8, ...)'にしてください」「エンコーダーの第二層目のカーネルサイズを5にしてください」などと記載し，GPT4oに修正を依頼することでコード生成しました，GPT4oはそれに合わせてエンコーダの残りの層やデコーダーも修正しました．

また，畳み込み層のフィルターは単純に縦，横，斜め方向の要素1で他は0というものではなく，エンコーダーの第一層には32個のフィルターが存在します．これらはどのようなフィルターか，GPT4oに尋ねてみましょう．

```
プロンプト「
以下のAutoencoderクラスのエンコーダーの第一層の畳み込み層におけるフィルターを画像として表示してください．
matplotlibを用いてください．

# コード
（Autoencoderクラスを貼り付ける）
」
```

![Autoencoderのエンコーダー部分の第一層の3x3のフィルターを濃淡図で示す．](image_keep/encoder0_filter.png){ width=7cm }

\label{fig:denoise_encoder_filter}

GPT4oが生成したコード(8000.2000.filter_vis.ipynb)を実行し
図\ref{fig:denoise_encoder_filter}に画像を示します．あまり直感的なフィルターでないことが分かります．
（これはモデルパラメタであり，特徴量週出のためのフィルターではない．）


## 実行

これから行う過程を図\ref{fig:add_denosise_function}に示します．
この図は，コードに追加された機能を視覚的に表しています．
背景が黒いノードは，ここで既に作成したコードが持つ機能を示しています．

![コードへの機能追加](image_keep/LLM-development-process-step3.png){ width=13cm }

\label{fig:add_denosise_function}


本筋に戻ります．img_filenameとして，ソルトアンドペッパーノイズを10パーミル加えた画像であるcropped_64x64_permil10/223_48_1.pngを使用した実行結果を示します（8000.1000.denose_code.ipynbをsetting=1として実行）
図\ref{fig:denoise_first_image}は，10エポック目と200エポック目での画像を表示しています．
10エポック目ではまだ濃淡見えないですが，学習が進むと濃淡がはっきりしてきます．

![10 エポック目での画像．](image_keep/image_223_48_1_no_range_mask/original_vs_denoised_image_from_223_48_1_png_epoch_10.png){ width=7cm }

![200 エポック目での画像．](image_keep/image_223_48_1_no_range_mask/original_vs_denoised_image_from_223_48_1_png_epoch_200.png){ width=7cm }

\caption{10 エポック目，200エポック目の画像}
\label{fig:denoise_first_image}



## 画素のヒストグラムと正規化変更

学習が進むにつれて濃淡が変化することが確認できました．
そこで，エポックごとに画素値の復元過程を観察するため，追加の指示としてトレーニングループに「一定のエポックごとに元画像とデノイズされた画像の画素値のヒストグラムを比較してください」とプロンプトに追加し，コードを再生成して実行しました（図\ref{fig:add_denosise_function}①），

![200 エポック目での画素値のヒストグラム．](image_keep/image_223_48_1_no_range_mask/DOS_from_223_48_3_png_epoch_200.png){ width=7cm }

\label{fig:denoise_dos_first_try}

図\ref{fig:denoise_dos_first_try}に200エポック目での画素値のヒストグラム（DOS）を示しました．
最初に確認すべきことでしたが，ソルトアンドペッパーノイズの影響でノイズ以外の画像が適切に正規化されていませんでした．
今回は，ソルトアンドペッパーノイズが画素のダイナミックレンジの最小値（0）または最大値（255）になるため，ノイズの影響を受けない画素は中央の値を持つことになります．
このため，ノイズの画素を特定することが簡単です．
これをGPT4oで実装します，maskの作成と，それに従ったコードの修正を依頼します．コードには大域変数mask_value0とmask_value1が関係するため，指示を詳細に書いています．
また，会話の回数が多い場合，コード生成時に前の文脈を忘れてしまうことがあるため，最後にコードを貼り付けることをお勧めします．

```
プロンプト「
{# コード}に以下の過程を加えてください．
{# 過程}に従って変更してください．動作するように他の部分も変更してください．
大域変数{# 変数}を定義してください．
mask作成はImageDatasetクラスが行ってください．

# 変数
論理型変数mask_value0を定義します．mask_value0=Trueとします．
論理型変数mask_value1を定義します．mask_value1=Trueとします．

# 過程
1. グレイスケール画像をファイルから読み込みます．画素値が0から1となる関数を用いずに画素値が0から255となる関数を用いて画像を読み込んでください．
   1.1. 画素値が0以下をmask0としてください．
   1.2. 画素値が255の画素をmask1してください．
   1.3. mask = mask0 | mask1 と論理和を取って定義します．

2. 画像を正規化します．
   2.1. 画像の最小値をmin_value，最大値をmax_valueとしてください．
   2.2. 画像からmask以外を選択してください．
   2.3. mask_value0がTrueの場合はmask0を除いた画像の最小値をmin_value，mask_value0がFalseの場合は画像全体の最小値をmin_valueとしてください．
   2.4. mask_value1がTrueの場合はmask1を除いた画像の最大値をmax_value，mask_value1がFalseの場合は画像全体の最大値をmax_valueとしてください．
   2.5. min_value，max_valueを用いて画像の正規化を行ってください．

# コード
（コードを貼りつけ）
」
```


![10 エポック目での画像．](image_keep/image_223_48_1/original_vs_denoised_image_from_223_48_1_png_epoch_10.png){ width=3cm }
![60 エポック目での画像．](image_keep/image_223_48_1/original_vs_denoised_image_from_223_48_1_png_epoch_60.png){ width=3cm }
![200 エポック目での画像．](image_keep/image_223_48_1/original_vs_denoised_image_from_223_48_1_png_epoch_200.png){ width=3cm }


![10 エポック目でのDOS．](image_keep/image_223_48_1/DOS_from_223_48_1_png_epoch_10.png){ width=3cm }
![60 エポック目でのDOS．](image_keep/image_223_48_1/DOS_from_223_48_1_png_epoch_60.png){ width=3cm }
![200 エポック目でのDOS．](image_keep/image_223_48_1/DOS_from_223_48_1_png_epoch_200.png){ width=3cm }


![エポック vs 損失関数](image_keep/image_223_48_1/epoch_vs_loss_from_223_48_1_png.png){ width=7cm }

\caption{上：10,60,200エポック目の元画像とデノイズされた画像，中：10,60,200エポック目の画素値のヒストグラム，下：エポック vs 損失関数}
\label{fig:image_dos_loss_second_try}




コードを再度作成し実行した結果を，図\ref{fig:image_dos_loss_second_try}に示します，図では，学習過程の画像，画素値のヒストグラム（DOS），エポックと損失関数の関係を示しています，（ソースコード8000.1000.denose_code.ipynbを`setting=2`として実行しています．）

図の上部には，10, 60, 200エポック目の元画像とデノイズ画像が示されており，中部にはそれぞれのピクセル値のヒストグラム，下部にはエポックと損失関数の関係が示されています．

まず，デノイズ画像を見てみましょう，10エポック目では濃淡がほぼありませんが，60エポック目では元画像に近くなり，ペッパーアンドソルトノイズも消去されています，200エポック目にはノイズがほぼ完全に消え，元画像に非常に近い画像となっています．

次に，ピクセル値のヒストグラムを確認します．
10エポック目はほぼ0.5の値を持ちますが，60エポック目では元画像のヒストグラムに近づき始めます．200エポック目には元画像にかなり近いヒストグラムが得られています．

最後に，エポックと損失関数の関係を見てみましょう．
50エポック目までは急激に損失が減少し，それ以降は減少の度合いが緩やかになります．
単一画像のみを用いて学習を行っているため，テストデータはありません．
訓練データの指標値のみを参照しており，エポック数を増やすほど過学習により損失関数が小さくなることから，元画像が出力されるようになります．

では，学習はいつ止めるのが適切でしょうか，

```
プロンプト「
デノイズオートエンコーダーを用いてデノイズをする場合にエポック数が多いと過学習しました，
学習をいつやめればいいですか？
」
```


と尋ねると，評価指標としてはMSEだけでなく，PSNRやSSIMを用いることを提案し，早期停止（Early Stopping）手法を紹介されました（図\ref{fig:add_denosise_function}②）．
先ほどは，なぜMAE以外の評価指標が必要なのか分かりませんでしたが，
これらは停止条件の評価指標として利用できるようです．

PSNR,SSIMに関してより詳しい説明を求めます．

```
プロンプト「
PSNR,SSIMを説明してください．

手法の概要

    手法を簡潔に説明してください．
    手法の目的を説明してください．
    手法の背景にある仮説を説明してください．

手法の詳細

    手法の概要・構成要素を説明してください．
    手法の概要・構成要素に紐づけられる基本的な数式や理論を説明してください．
    手法の長所，短所を説明してください．
    手法の実行で得られる結果をどのように評価するかを説明してください．
    コードの説明ではなく手法について教えてください．

関連手法

    手法の上位概念と類似手法を階層構造を用いて説明してください．
    手法の目的とその目的を達成する類似手法を階層構造を用いて説明してください．
    関連する他の手法との違いと選択基準について説明してください．
」
```

\noindent
回答をまとめると，どちらも元画像と変更された画像との画像劣化を評価する手法です．
伝統的な計測法にPSNRやMSEがあり，人間視覚モデルに基づく計測法にSSIMがあります，指標の選択基準としては:

- PSNR: 計算が簡単で広く使われていますが，人間の視覚特性を考慮していないため，視覚的に劣る画像でも高い数値を示すことがあります．PSNRの値が大きいほど画像が似ており，30dB以上が良，40dB以上が非常に良いとされます．
- SSIM: 人間の視覚特性を考慮しており，画像の構造的な違いに敏感で視覚的品質評価には優れていますが，計算が複雑です．SSIMは0から1までの値を取り，1に近いほど2つの画像が構造的に似ていることを示します．


これらの評価指標を加えることにします．
プロンプトの学習後部分に
「最後にエポック vs PSNR, エポック vs SSIMを図示してください．」
と追加しコードを生成し
実行させます．
図\ref{fig:epoch_psnr_ssim}にエポックとそれらの評価指標の関係を示します．エポック vs MSEを上下反転したような図が得られました．
PSNRはskimage.metricsパッケージを用いており，単位はdBです．MSE損失関数と同じく，50エポック目までとその後で傾きが異なる評価指標が得られました．



![epoch vs PSNR](image_keep/image_223_48_1/epoch_vs_PSNR_from_223_48_1_png.png){ width=7cm }

![epoch vs SSIM](image_keep/image_223_48_1/epoch_vs_SSIM_from_223_48_1_png.png){ width=7cm }

\caption{エポック vs PSNR, エポック vs SSIM.}
\label{fig:epoch_psnr_ssim}



停止条件についてを再び尋ねます．（図\ref{fig:add_denosise_function}②）

```
プロンプト「
一画像をデノイズオートエンコーダーを用いてデノイズをする場合にエポック数が多いと過学習しました．一画像なので検証用のデータはありません．MSE, PSNR, SSIMを計算しました
学習をいつ止めればいいですか？
」
```

---（回答まとめ）---

\noindent
同じチャットでのやり取りでも，「一画像なので検証用データが無い」と明示しなければ，検証用データについての言及があることが多いため，注意が必要です．
GPT4oの回答による学習をやめるタイミングの案は以下のとおりです．

- MSE（Mean Squared Error）:
損失関数として使用する場合，トレーニング中にMSEの減少が止まるか，増加し始めたタイミングで学習を停止する．
- PSNR（Peak Signal-to-Noise Ratio）:
PSNRは高いほど良いので，PSNRが増加しなくなった時点，または下降し始めた時点で学習を停止する．
- SSIM（Structural Similarity Index）:
SSIMも高いほど理想的なので，SSIMの増加が止まるか，減少し始めた時点で学習を停止する．

\noindent
例えば， patienceを使って，連続して複数エポック（例えば，
5エポック）の間，変化量が少ない状態が続いたなら学習を停止することで，過学習を防ぎつつ，最適なパフォーマンスのモデルを生成することができる．

---（回答まとめここまで）---

各種評価指標値の変化量が少なくなるエポック数は約50〜60 エポック程度に見えます．実際にこのエポックではソルトアンドペッパーノイズは消えています．


## マスクの追加

60エポック目では元画像よりもなめらかに変化しているように見えます．デノイズと平滑化を同時に行った結果とも言えるでしょう．
人の目にはこの処理で十分かもしれません，しかし，画素値のヒストグラムは元画像と大きく異なっており，まだ改良の余地があります．

Noise2Selfアルゴリズムの特徴として，

---（回答まとめ）---

1. 画像の一部をランダムに隠して学習しない．
2. マスクされていない部分から損失関数を計算する．
3. マスクされていない部分のみからマスクされた部分の画素値を予測する．

\noindent という3つの要素があるとGPT4oは説明しました．

---（回答まとめここまで）---

ここでは，これがNoise2Selfかは議論せず，似たアプローチを試してみましょう（図\ref{fig:add_denosise_function}④）．

ソルトアンドペッパーノイズの画素はすでに特定しています．このマスクを利用して，最も簡単な手法である2.と3.の実装をGPT4oに依頼します．


```
プロンプト「
{# コード}に以下の過程を加えてください．
{# 過程}に従って変更してください．動作するように他の部分も変更してください．
大域変数{# 変数}を定義してください．

# 変数
論理型変数criterion_maskを定義します．criterion_mask=Trueとします．
論理型変数image_maskを定義します．image_mask=Falseとします．

# 過程
1. image_maskがTrueの場合はmask部分の値を変更します．
   1.1. 画像からmask以外を選択してください．
   1.2. そのmedianを計算します．
   1.3. 画像のmask部分にmedian値を代入してください．

2. criterion_maskがTrueの場合はmask部分を損失関数に含めないでください．

# コード
（コードを貼りつけ）
」
```

デフォルトでは全体の評価だった損失を画素レベルで計算するために以下の設定を行っています．
```
criterion = nn.MSELoss(reduction='none')
```
\noindent
また，マスク外の画素のみで平均化するために，
```
loss.sum(dim=(1, 2, 3))
```
\noindent
として和を取っています．

最後はユーザー，もしくはGPT4oのプロンプト経由で細かいコード修正を行っています．
（図\ref{fig:add_denosise_function}⑤）
image_maskがTrueの場合はmaskされたimageの画素値をmedian値に修正しています．
そして，criterion_maskがTrueの場合は以下のMSE 損失関数(loss)を計算します．
$$
\text{loss} = \sum_{ij} (X_{ij} - \hat{X}_{ij})^2* (1-\text{mask}_{ij})
$$
ここで$X$は正規化した入力画像，$\hat{X}$はデノイズをした画像（予測値），マスク（変数mask）は1（マスクする）もしくは0（マスクしない）の値を取ります．
それぞれ画素$i,j$に対して定義されます．

setting3（criterion_mask=True, image_mask=False, mask_value0=True, mask_value1=True）とした場合の学習結果を図\ref{fig:denoise_final_image_dos}に示します．
（8000.1000.denose_code.ipynbをsetting=3として実行)

この設定では，マスクされていない部分から損失関数を計算することで，ソルトアンドペッパーノイズの復元（過学習）をほぼ無くすことができました．
ただし，この方法はすべての画像で効果があるわけではありません．
同コードでimage_mask=Trueとすると，さらにソルトアンドペッパーノイズを効果的に消去できます．
GPT4oに尋ねると，ソルトアンドペッパーノイズがあると分かっている場合，メディアンフィルタなどをニューラルネットワークの学習前に適用することも勧められました．



![損失関数にマスクを加えた画像](image_keep/image_223_48_1_criterion_mask/original_vs_denoised_image_from_223_48_1_png_epoch_200.png){ width=7cm }

![損失関数にマスクを加えた画素値のヒストグラム](image_keep/image_223_48_1_criterion_mask/DOS_from_223_48_1_png_epoch_200.png){ width=3cm }

\caption{損失関数にマスクを加えた200 エポック目の画像と画素値のヒストグラム．}
\label{fig:denoise_final_image_dos}


## 最後に

現在（2024年10月)はもし，ユーザーにある程度知識があれば，
GPTが現れる前に初学者向けの書籍に書かれているであろう内容をGPTから学習し，コード生成を行うことが可能になっています．
ユーザーがGPT4oに多くの指示を行えるようになるほどGPT4oは適切なコードを生成できます．
特に，ニューラルネットワークを用いたコードの雛形作成や，複雑なパラメータの整合をとる作業をGPT4oに任せると，人間が行う手間を大幅に省くことができます．

しかし，完全にGPT4oに任せられるようにはなっていません．
GPT4oは簡単な事柄や例が多い技術に対しては妥当な説明を行うことが多いのですが，
誤った，もしくは簡単すぎて誤解する説明を行うこともあります．
コードに関しては既に説明した誤りがよく有ります．
例えば，Pytorchのテンソルや入力画像と出力画像の値の範囲を間違えることは多くあります．
しかし，コードの不明事項をユーザーに尋ねるようにGPT4oに依頼しても些細な不明点を全て尋ねていたらきりがありません．GPT4oが動作するコードを作ることを優先するとある程度の誤りは仕方が無いことかもしれません．

また，GPT4oはデコーダー部分で'nn.Conv2d'を使うAutoencoderが生成しました．通常はデコーダー部分は'nn.ConvTranspose2d'を使うのですが，この場合は解像度の変更をしないので'nn.Conv2d'を用いても動作します．これは私が画像を(64,64)と指定したせいかもしれません．

画像処理では，GPT4oがPSNRやSSIMを評価指標として挙げたのだと思います．一方，ユーザーが画素値ヒストグラムを視覚的な評価指標として用いたのですが，ヒストグラムからエントロピーをscipy.stats.entropyで計算する手法を推薦することはほぼありませんでした．

今後のチャットGPTを含むLLMの発展により，状況が大きく変わるかもしれませんが，**ユーザーはLLMからの回答を鵜呑みにせず，批判的に評価し，結果を常に検証することが重要です．** GPT4oを学習支援ツールとして活用し，本書の読者がGPT4oを超える知識を持ち，思考できるようになることを期待します．
